# Задание 7. Максимальный поток (Форд–Фалкерсон / Эдмондс–Карп)

Исток: **1**, сток: **8**.

In [15]:
# Импорт библиотек
from collections import deque, defaultdict
import os
import pandas as pd
import matplotlib.pyplot as plt

In [16]:
# Дуги графа и их пропускные способности
# Формат: (начало, конец): пропускная способность

capacity = {
    (1, 3): 95,
    (1, 4): 75,
    (1, 5): 57,
    (1, 2): 32,

    (2, 3): 5,
    (2, 5): 23,
    (2, 8): 16,

    (3, 4): 18,
    (3, 6): 6,

    (4, 6): 9,
    (4, 5): 24,

    (6, 5): 11,
    (6, 7): 7,

    (5, 7): 20,
    (5, 8): 94,

    (7, 8): 81,
}

source = 1
sink = 8

vertices = sorted(set([u for u, v in capacity] + [v for u, v in capacity]))

print("Вершины:", vertices)
print("Исток:", source)
print("Сток:", sink)
print("Количество дуг:", len(capacity))

Вершины: [1, 2, 3, 4, 5, 6, 7, 8]
Исток: 1
Сток: 8
Количество дуг: 16


In [17]:
# Таблица исходных дуг

initial_df = pd.DataFrame(
    [{"Дуга": f"{u} → {v}", "Пропускная способность": c}
     for (u, v), c in capacity.items()]
)

initial_df

,Дуга,Пропускная способность
0,1 → 3,95
1,1 → 4,75
2,1 → 5,57
3,1 → 2,32
4,2 → 3,5
5,2 → 5,23
6,2 → 8,16
7,3 → 4,18
8,3 → 6,6
9,4 → 6,9


In [18]:
# Функции алгоритма

def build_residual_graph(flow):
    """Создаёт остаточную сеть по текущему потоку."""
    residual = defaultdict(dict)

    for (u, v), c in capacity.items():
        forward_residual = c - flow[(u, v)]
        if forward_residual > 0:
            residual[u][v] = forward_residual

        if flow[(u, v)] > 0:
            residual[v][u] = flow[(u, v)]

    return residual


def find_augmenting_path_bfs(flow):
    """Ищет увеличивающий путь из source в sink в остаточной сети."""
    residual = build_residual_graph(flow)

    parent = {source: None}
    q = deque([source])

    while q:
        u = q.popleft()
        for v, residual_capacity in residual[u].items():
            if v not in parent and residual_capacity > 0:
                parent[v] = u
                if v == sink:
                    path = []
                    cur = sink
                    while cur is not None:
                        path.append(cur)
                        cur = parent[cur]
                    path.reverse()

                    delta = min(residual[path[i]][path[i+1]] for i in range(len(path)-1))
                    return path, delta, residual

                q.append(v)

    return None, 0, residual


def residual_table(residual):
    """Таблица остаточных дуг."""
    rows = []
    for u in sorted(residual):
        for v in sorted(residual[u]):
            rows.append({
                "Остаточная дуга": f"{u} → {v}",
                "Остаточная пропускная способность": residual[u][v]
            })
    return pd.DataFrame(rows)


def flow_table(flow):
    """Таблица текущих потоков только по исходным дугам."""
    rows = []
    for (u, v), c in capacity.items():
        rows.append({
            "Дуга": f"{u} → {v}",
            "Поток": flow[(u, v)],
            "Пропускная способность": c,
            "Остаток": c - flow[(u, v)]
        })
    return pd.DataFrame(rows)


def value_of_flow(flow):
    """Величина потока: сумма потоков из истока."""
    return sum(flow[(u, v)] for (u, v) in capacity if u == source)

In [19]:
# Начальный поток равен нулю по всем дугам

flow = defaultdict(int)

print("Начальный поток:")
display(flow_table(flow))


Начальный поток:


,Дуга,Поток,Пропускная способность,Остаток
0,1 → 3,0,95,95
1,1 → 4,0,75,75
2,1 → 5,0,57,57
3,1 → 2,0,32,32
4,2 → 3,0,5,5
5,2 → 5,0,23,23
6,2 → 8,0,16,16
7,3 → 4,0,18,18
8,3 → 6,0,6,6
9,4 → 6,0,9,9


In [20]:
# Запуск алгоритма Форда-Фалкерсона

step = 1
history = []

while True:
    path, delta, residual = find_augmenting_path_bfs(flow)

    if path is None:
        print("Увеличивающих путей больше нет.")
        print("Алгоритм завершён.")
        break

    print("=" * 80)
    print(f"ШАГ {step}")
    print("Найден увеличивающий путь:")
    print(" → ".join(map(str, path)))
    print("Минимальная остаточная пропускная способность на пути:")
    print("Δ =", delta)

    print("\nОстаточная сеть ДО увеличения потока:")
    display(residual_table(residual))

    for u, v in zip(path, path[1:]):
        if (u, v) in capacity:
            flow[(u, v)] += delta
        else:
            flow[(v, u)] -= delta

    current_value = value_of_flow(flow)

    print("\nПоток ПОСЛЕ увеличения:")
    display(flow_table(flow))

    print("Текущая величина потока:")
    print("|f| =", current_value)

    history.append({
        "Шаг": step,
        "Путь": " → ".join(map(str, path)),
        "Δ": delta,
        "Величина потока после шага": current_value
    })

    step += 1

print("=" * 80)
print("ИТОГОВАЯ ВЕЛИЧИНА МАКСИМАЛЬНОГО ПОТОКА:")
print("f_max =", value_of_flow(flow))

ШАГ 1
Найден увеличивающий путь:
1 → 5 → 8
Минимальная остаточная пропускная способность на пути:
Δ = 57

Остаточная сеть ДО увеличения потока:


,Остаточная дуга,Остаточная пропускная способность
0,1 → 2,32
1,1 → 3,95
2,1 → 4,75
3,1 → 5,57
4,2 → 3,5
5,2 → 5,23
6,2 → 8,16
7,3 → 4,18
8,3 → 6,6
9,4 → 5,24



Поток ПОСЛЕ увеличения:


,Дуга,Поток,Пропускная способность,Остаток
0,1 → 3,0,95,95
1,1 → 4,0,75,75
2,1 → 5,57,57,0
3,1 → 2,0,32,32
4,2 → 3,0,5,5
5,2 → 5,0,23,23
6,2 → 8,0,16,16
7,3 → 4,0,18,18
8,3 → 6,0,6,6
9,4 → 6,0,9,9


Текущая величина потока:
|f| = 57
ШАГ 2
Найден увеличивающий путь:
1 → 2 → 8
Минимальная остаточная пропускная способность на пути:
Δ = 16

Остаточная сеть ДО увеличения потока:


...

Текущая величина потока:
|f| = 120
ШАГ 8
Найден увеличивающий путь:
1 → 4 → 6 → 5 → 7 → 8
Минимальная остаточная пропускная способность на пути:
Δ = 8

Остаточная сеть ДО увеличения потока:


,Остаточная дуга,Остаточная пропускная способность
0,1 → 3,89
1,1 → 4,50
2,2 → 1,32
3,2 → 3,5
4,2 → 5,7
5,3 → 1,6
6,3 → 4,18
7,4 → 1,25
8,4 → 6,8
9,5 → 1,57



Поток ПОСЛЕ увеличения:


,Дуга,Поток,Пропускная способность,Остаток
0,1 → 3,6,95,89
1,1 → 4,33,75,42
2,1 → 5,57,57,0
3,1 → 2,32,32,0
4,2 → 3,0,5,5
5,2 → 5,16,23,7
6,2 → 8,16,16,0
7,3 → 4,0,18,18
8,3 → 6,6,6,0
9,4 → 6,9,9,0


Текущая величина потока:
|f| = 128
Увеличивающих путей больше нет.
Алгоритм завершён.
ИТОГОВАЯ ВЕЛИЧИНА МАКСИМАЛЬНОГО ПОТОКА:
f_max = 128


In [21]:
# Краткая таблица всех шагов

history_df = pd.DataFrame(history)
history_df

,Шаг,Путь,Δ,Величина потока после шага
0,1,1 → 5 → 8,57,57
1,2,1 → 2 → 8,16,73
2,3,1 → 4 → 5 → 8,24,97
3,4,1 → 2 → 5 → 8,13,110
4,5,1 → 3 → 6 → 7 → 8,6,116
5,6,1 → 4 → 6 → 7 → 8,1,117
6,7,1 → 2 → 5 → 7 → 8,3,120
7,8,1 → 4 → 6 → 5 → 7 → 8,8,128
